In [ ]:
import os
import json
import base64
import io
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from groq import Groq
from dotenv import load_dotenv

# Força o Matplotlib a rodar em modo "Headless" (sem interface gráfica)
matplotlib.use('Agg') 

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Configuração de Estilo Visual (Branding Primordial Data)
sns.set_theme(style="white")
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['#3b82f6', '#10b981', '#f59e0b', '#ef4444'])
plt.rcParams['font.family'] = 'sans-serif'

In [ ]:
def planejar_grafico(contexto_clinico, pergunta_medico):
    """
    IA analisa os dados e decide a melhor forma de visualização.
    """
    prompt = f"""
    SISTEMA DE ANÁLISE CLÍNICA - PRIMORDIAL DATA
    DADOS DISPONÍVEIS: {contexto_clinico}
    PERGUNTA: {pergunta_medico}
    
    INSTRUÇÕES:
    1. Analise se os dados são suficientes para responder.
    2. Escolha o melhor gráfico: 'bar' (categorias), 'pie' (proporções), 'line' (tempo), 'scatter' (correlação).
    3. Responda APENAS um JSON puro, sem markdown ou textos explicativos.

    FORMATO DE RESPOSTA:
    {{
      "analise": "Sua conclusão técnica aqui",
      "tipo_grafico": "bar | pie | line | scatter",
      "titulo": "Título do Gráfico",
      "eixo_x": ["Label1", "Label2"],
      "valores": [10, 20]
    }}
    """
    
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        
        texto_ia = response.choices[0].message.content
        
        # Extrator robusto de JSON
        inicio = texto_ia.find('{')
        fim = texto_ia.rfind('}') + 1
        return json.loads(texto_ia[inicio:fim])
        
    except Exception as e:
        return {
            "analise": f"Erro na análise neural: {str(e)}",
            "tipo_grafico": "bar",
            "titulo": "Erro de Processamento",
            "eixo_x": ["Erro"],
            "valores": [0]
        }

In [ ]:
def gerar_imagem_grafico(plano):
    """
    Recebe o plano da IA e desenha o gráfico de forma profissional.
    """
    # Criando a figura
    fig, ax = plt.subplots(figsize=(10, 5))
    
    tipo = plano.get('tipo_grafico', 'bar').lower()
    labels = plano.get('eixo_x', [])
    valores = plano.get('valores', [])
    titulo = plano.get('titulo', 'Análise Primordial')

    if tipo == 'pie':
        ax.pie(valores, labels=labels, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("muted"))
    elif tipo == 'line':
        ax.plot(labels, valores, marker='o', linewidth=3, color='#3b82f6')
        ax.fill_between(labels, valores, alpha=0.1, color='#3b82f6')
    elif tipo == 'scatter':
        ax.scatter(labels, valores, s=150, alpha=0.6)
    else: # Default: Bar
        bars = ax.bar(labels, valores, color='#3b82f6', edgecolor='white')
        ax.bar_label(bars, padding=3, fontsize=10)

    # Limpeza de layout (Design de BI)
    ax.set_title(titulo, fontsize=14, fontweight='bold', pad=20)
    sns.despine(left=True, bottom=False)
    plt.tight_layout()

    # Conversão para Base64 para o HTML
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=120)
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    
    return f"data:image/png;base64,{img_b64}"

In [ ]:
# Simulação de dados que viriam do database.py
dados_ficticios = "Paciente 1: Masculino, 45 anos, Diabetes. Paciente 2: Feminino, 30 anos, Hipertensão."
pergunta = "Qual a proporção de doenças por gênero?"

plano = planejar_grafico(dados_ficticios, pergunta)
print("IA escolheu o gráfico:", plano['tipo_grafico'])
print("Análise:", plano['analise'])

# Se estiver no Jupyter, você pode visualizar assim:
# from IPython.display import Image
# img_url = gerar_imagem_grafico(plano)
# print("Sucesso: Gráfico gerado!")